# 02. Data Cleaning & Exploratory Data Analysis
**COVID-19 Data Tracking System**

**Objectives:**
- Clean the raw dataset
- Handle missing values
- Remove aggregated regions (World, continents, income groups)
- Create useful analytical features
- Perform initial EDA
- Save the cleaned dataset

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

# Paths
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "owid-covid-data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load data
df = pd.read_csv(RAW_DATA_PATH)
print(f"Raw data loaded: {df.shape}")

Libraries imported successfully!
Raw data loaded: (617667, 61)


In [4]:
print("Shape:", df.shape)
print("\nMissing values (top 15):")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nData types:")
print(df.dtypes)

Shape: (617667, 61)

Missing values (top 15):
human_development_index                    617667
weekly_icu_admissions                      606674
weekly_icu_admissions_per_million          606674
excess_mortality_cumulative_per_million    603823
excess_mortality                           603789
excess_mortality_cumulative                603789
excess_mortality_cumulative_absolute       603789
weekly_hosp_admissions_per_million         593170
weekly_hosp_admissions                     593170
icu_patients                               578551
icu_patients_per_million                   578551
hosp_patients_per_million                  577011
hosp_patients                              577011
total_boosters                             558883
total_boosters_per_hundred                 558883
dtype: int64

Data types:
country                           str
date                              str
total_cases                   float64
new_cases                     float64
new_cases_smoothed        

In [5]:
# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

# Sort data
df = df.sort_values(['country', 'date']).reset_index(drop=True)

print("Date converted successfully.")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

Date converted successfully.
Date range: 2020-01-01 → 2026-08-30


In [6]:
# These are not real countries
exclude_list = [
    'World', 'Europe', 'Asia', 'Africa', 'North America', 'South America', 'Oceania',
    'European Union', 'High income', 'Upper middle income', 'Lower middle income',
    'Low income', 'International'
]

# Keep only real countries (those that have a continent)
df_clean = df[df['continent'].notna()].copy()

# Extra safety: remove any remaining aggregates
df_clean = df_clean[~df_clean['country'].isin(exclude_list)]

print(f"After removing aggregates: {df_clean.shape}")
print(f"Unique countries remaining: {df_clean['country'].nunique()}")

After removing aggregates: (572427, 61)
Unique countries remaining: 239


In [7]:
# Key numerical columns
key_cols = [
    'total_cases', 'new_cases', 'total_deaths', 'new_deaths',
    'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated',
    'population'
]

# Fill missing values in cumulative columns with 0 (logical for early pandemic)
for col in ['total_cases', 'total_deaths', 'total_vaccinations', 
            'people_vaccinated', 'people_fully_vaccinated']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

# New cases / new deaths → fill with 0
for col in ['new_cases', 'new_deaths']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

print("Missing values handled for key columns.")
print(df_clean[key_cols].isnull().sum())

Missing values handled for key columns.
total_cases                   0
new_cases                     0
total_deaths                  0
new_deaths                    0
total_vaccinations            0
people_vaccinated             0
people_fully_vaccinated       0
population                 2479
dtype: int64


In [8]:
# 1. Case Fatality Rate (CFR) %
df_clean['case_fatality_rate'] = np.where(
    df_clean['total_cases'] > 0,
    (df_clean['total_deaths'] / df_clean['total_cases']) * 100,
    0
)

# 2. Cases per million (already exists, but we keep it)
# 3. Deaths per million (already exists)

# 4. Vaccination coverage (%)
df_clean['vaccination_rate'] = np.where(
    df_clean['population'] > 0,
    (df_clean['people_vaccinated'] / df_clean['population']) * 100,
    0
)

df_clean['fully_vaccinated_rate'] = np.where(
    df_clean['population'] > 0,
    (df_clean['people_fully_vaccinated'] / df_clean['population']) * 100,
    0
)

# 5. 7-day rolling average of new cases and new deaths
df_clean['new_cases_7day_avg'] = df_clean.groupby('country')['new_cases'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

df_clean['new_deaths_7day_avg'] = df_clean.groupby('country')['new_deaths'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

print("New features created successfully!")
print(df_clean[['case_fatality_rate', 'vaccination_rate', 'fully_vaccinated_rate', 
                'new_cases_7day_avg', 'new_deaths_7day_avg']].describe())

New features created successfully!
       case_fatality_rate  vaccination_rate  fully_vaccinated_rate  \
count           572427.00         572427.00              572427.00   
mean                 7.60              5.72                   4.96   
std                689.78             19.06                  17.38   
min                  0.00              0.00                   0.00   
25%                  0.33              0.00                   0.00   
50%                  0.92              0.00                   0.00   
75%                  1.89              0.00                   0.00   
max             103875.00            112.08                 110.19   

       new_cases_7day_avg  new_deaths_7day_avg  
count           572427.00            572427.00  
mean              1361.38                12.44  
std              26491.80                97.76  
min                  0.00                 0.00  
25%                  0.00                 0.00  
50%                  0.57               

In [9]:
print("="*60)
print("CLEANED DATASET SUMMARY")
print("="*60)
print(f"Shape                     : {df_clean.shape}")
print(f"Unique countries          : {df_clean['country'].nunique()}")
print(f"Date range                : {df_clean['date'].min().date()} → {df_clean['date'].max().date()}")
print(f"Total records             : {len(df_clean):,}")

print("\nColumns now available:")
print(df_clean.columns.tolist())

CLEANED DATASET SUMMARY
Shape                     : (572427, 66)
Unique countries          : 239
Date range                : 2020-01-01 → 2026-08-30
Total records             : 572,427

Columns now available:
['country', 'date', 'total_cases', 'new_cases', 'new_cases_smoothed', 'total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million', 'total_deaths', 'new_deaths', 'new_deaths_smoothed', 'total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million', 'excess_mortality', 'excess_mortality_cumulative', 'excess_mortality_cumulative_absolute', 'excess_mortality_cumulative_per_million', 'hosp_patients', 'hosp_patients_per_million', 'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million', 'icu_patients', 'icu_patients_per_million', 'weekly_icu_admissions', 'weekly_icu_admissions_per_million', 'stringency_index', 'reproduction_rate', 'total_tests', 'new_tests', 'total_tests_per_thousand', 'new_tests_per_thousand', 'new_tests_smoothed'

In [10]:
# Save as CSV
output_path = PROCESSED_DIR / "covid_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"✅ Cleaned data saved successfully to:\n{output_path}")
print(f"File size: {output_path.stat().st_size / (1024*1024):.2f} MB")

✅ Cleaned data saved successfully to:
d:\OneDrive\Documents\Desktop\Resume Projects\covid19-data-tracking-system\data\processed\covid_cleaned.csv
File size: 198.64 MB


In [11]:
# Top 10 countries by total cases (latest available)
latest = df_clean.sort_values('date').groupby('country').tail(1)

print("Top 10 countries by Total Cases (latest data):")
display(latest[['country', 'total_cases', 'total_deaths', 'case_fatality_rate', 'fully_vaccinated_rate']]
        .sort_values('total_cases', ascending=False)
        .head(10)
        .reset_index(drop=True))

Top 10 countries by Total Cases (latest data):


,country,total_cases,total_deaths,case_fatality_rate,fully_vaccinated_rate
0,United States,103436829.00,1239049.00,1.20,0.00
1,China,99381761.00,122398.00,0.12,0.00
2,India,45056221.00,533849.00,1.18,0.00
3,France,39061500.00,168207.00,0.43,0.00
4,Germany,38437953.00,174979.00,0.46,0.00
5,Brazil,38050567.00,704066.00,1.85,0.00
6,South Korea,34571873.00,35934.00,0.10,0.00
7,Japan,33803572.00,74694.00,0.22,0.00
8,Italy,26969918.00,198523.00,0.74,0.00
9,United Kingdom,25122084.00,232112.00,0.92,0.00
